In [112]:
############################################################
#read in setup -- kind of stupid system actually
############################################################

import pandas as pd
from datetime import datetime, timedelta






with open('setup.py') as f:
    code = f.read()
exec(code)
run_sql("use hoodalgo_db")












#Create/reset driver
#################################################################################################################
try:
    driver.close()
except:
    pass

try:
    driver.quit()
except:
    pass

import time, os

time.sleep(.05)

# SAFE CLEAN (won’t kill your real Chrome)
os.system("pkill -f chromedriver")
os.system("pkill -f 'chrome.*--remote-debugging-port'")
os.system("pkill -f 'chrome.*--user-data-dir=/Users/deanemarks/selenium_chrome_profile'")

time.sleep(.05)

# Recreate driverD
driver = create_driver_profile_1()

time.sleep(.05)

# Load page
file_path = "file://" + base_dir + "templates/loading_page.html"
driver.get(file_path)

driver.execute_script("document.body.style.zoom='100%'")
time.sleep(2)
#################################################################################################################










#EXECUTE in MODUELS/ FUNCTIONS -  MAKES SHARED NAMESPACE
############################################################
with open('functions.py') as f:
    code = f.read()
exec(code)
############################################################
















#Update Robinhood Universe Every Week
#################################################################################################################
file_path = 'robinhood_universe.csv'
created_ts = os.path.getctime(file_path)
universe_created_dt = datetime.fromtimestamp(created_ts)
one_week_ago = datetime.now() - timedelta(days=7)


if universe_created_dt > one_week_ago:
    number_of_days = str(universe_created_dt - datetime.now()).split(",")[0].replace('-','').replace(' day','')
    update_loading_page(driver, f"Robinhood Universe Scraped {number_of_days} Day Ago - Dont Rescrape")
    time.sleep(1.5)


if universe_created_dt < one_week_ago:
    update_loading_page(driver, "Scraping Full Robinhood Equities Universe")
    scrape_robinhood_universe(num_workers = 20)
    update_loading_page(driver, "Robinhood Equities Universe successfully Scrapped")
    time.sleep(1.5)
#################################################################################################################
















#Fetch Fundamentals Data and filter every 10 minutes: 
#################################################################################################################

update_fundamentals_time_delta = 3


file_path = base_dir + "robinhood_universe_filtered.csv"
created_ts = os.path.getctime(file_path)
filtered_universe_ran_time = datetime.fromtimestamp(created_ts)
filtered_universe_run_again_time = filtered_universe_ran_time + timedelta(minutes=update_fundamentals_time_delta)




if os.path.exists(file_path):


    #If PAST  RUN_AGAIN_TIME example 
    if datetime.now() > filtered_universe_run_again_time:
        print(f'not within the {update_fundamentals_time_delta} minute window/n scraping all tickers then will refilter')

        
        df = pd.read_csv('robinhood_universe.csv')
        stock_universe_df = df[
            (df['tradeable'] == True) &
            (df['state'] == 'active') &
            (df['type'] == 'stock') &  # 👈 this is what you're missing
            #(df['all_day_tradability'] == 'tradable') &
            (df['fractional_tradability'] == 'tradable')
        ]
        stock_universe_list = stock_universe_df['symbol'].to_list()
        
        #UPDATE GUI NOTE
        update_loading_page(driver, f"NOW > RUN_AGAIN_TIME - Rescrape full Robinhood_universe  ")
        time.sleep(1.1)
        update_loading_page(driver, f"Fetching Robinhood Fundamentals Data - {len(stock_universe_list)} stocks ")


        
        #SCRAPE FUNDAMENTASL FX 
        robinhood_fundamentals_df = fetch_robinhood_fundamentals(ticker_list = stock_universe_list,chunk_size = 100)

        
        #UPDATE GUI NOTE
        update_loading_page(driver, "Scrape Complete ")
        time.sleep(1)


        #Fundamental Filter (DO NOT OVERFILTER HERE)
        ####################
        
        #Pre-Calculate Metrics
        df = robinhood_fundamentals_df.copy()
        df['average_volume_30_days'] = df['average_volume_30_days'].replace(0, 1)
        df['open'] = df['open'].replace(0, 1)
        
        # NO rel_volume filter here
        # NO range_pct filter here
        
        fundamentals_filter_df = df[
        
            # =========================
            # PRICE RANGE (stable)
            # =========================
            (df['open'] >= 1.5) &
            (df['open'] <= 30) &
        
            # =========================
            # SIZE (stable)
            # =========================
            (df['market_cap'] >= 20_000_000) &
            (df['market_cap'] <= 10_000_000_000) &
        
            # =========================
            # FLOAT (stable)
            # =========================
            (df['shares_float'] >= 1_000_000) &
        
            # =========================
            # BASE LIQUIDITY (loose)
            # =========================
            (df['average_volume_30_days'] >= 150_000)
        
        ]
        ####################

        
        #Filter tickers update load page
        filtered_tickers = fundamentals_filter_df['ticker'].dropna().unique().tolist()
        update_loading_page(driver, f"Done - {len(filtered_tickers)} stocks passed Fundamental filter")
        time.sleep(1)
        fundamentals_filter_df.to_csv("robinhood_universe_filtered.csv")






    
    
        
    #If INSIDE RUN_AGAIN TIME 
    if datetime.now() < filtered_universe_run_again_time:
        print('is within the 30 minute window/n scraping only the filtered tickers then will do all after 30 minutes')

        filtered_universe_df = pd.read_csv("robinhood_universe_filtered.csv")
        filtered_universe_list = filtered_universe_df['ticker'].to_list()



        #UPDATE GUI NOTE
        update_loading_page(driver, f"NOW < RUN_AGAIN_TIME Scraping filtered_robinhood_universe  ")
        time.sleep(1.1)
        update_loading_page(driver, f"Fetching Robinhood Fundamentals Data - {len(filtered_universe_list)} stocks ")


        

        #filter the tickers here. 
        robinhood_fundamentals_df = fetch_robinhood_fundamentals(ticker_list = filtered_universe_list,chunk_size = 100)
        filtered_tickers = robinhood_fundamentals_df['ticker'].dropna().unique().tolist()
        update_loading_page(driver, f"Complete ")

        

    
else:
    print("File does NOT exist")
    robinhood_fundamentals_df = fetch_robinhood_fundamentals(ticker_list = stock_universe_list,chunk_size = 100)
    robinhood_fundamentals_df.to_csv("robinhood_universe_filtered.csv")
    print('saved')


#################################################################################################################




    










✅ Deane’s MySQL Connector V39 — has query_value function 
Imported Selenium engine -- v15
📦 Switched default DB to: new_algo_db
📦 Switched default DB to: hoodalgo_db
functions read in 
not within the 3 minute window/n scraping all tickers then will refilter


/var/folders/jz/pvmnjkqd24dg39kvlscdc7bc0000gn/T/ipykernel_20447/3021522680.py:154: DtypeWarning: Columns (34) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('robinhood_universe.csv')


scraping robinhood fundamentals
🧲 Fetching 4212 fundamentals
✅ done: 4210 rows


In [107]:
filtered_tickers = filtered_tickers[:200]

In [69]:
len(stock_twits_analysis_df)

58

In [108]:
#Scrape Stocktwits W Selenium Threads (5 Workers, DF Safe)
#############################################################################################################


start = time.time()

import os
import time
import pandas as pd
from concurrent.futures import ThreadPoolExecutor

# --- HARD RESET (light) ---
try:
    os.system("pkill -f chromedriver")
except:
    pass

time.sleep(1)


# --- CONFIG ---
num_workers = 5


# --- SPLIT TICKERS ---
ticker_lists = [filtered_tickers[i::num_workers] for i in range(num_workers)]


# --- CREATE DRIVERS ---
drivers = []

i = 0
while i < num_workers:

    try:
        profile_name = "selenium_chrome_profile" if i == 0 else f"selenium_chrome_profile_{i+1}"
        driver = create_driver_profile(profile_name)
        drivers.append(driver)
    except:
        drivers.append(None)

    i += 1


# --- RUN THREADS ---
dfs = []

with ThreadPoolExecutor(max_workers=num_workers) as executor:

    futures = []

    i = 0
    while i < num_workers:

        if drivers[i] is not None:
            futures.append(
                executor.submit(
                    selenium_fetch_stocktwits_sentiment,
                    ticker_lists[i],
                    drivers[i]
                )
            )

        i += 1

    # collect DATAFRAMES (NOT extend)
    i = 0
    while i < len(futures):

        result = futures[i].result()

        if result is not None and len(result) > 0:
            dfs.append(result)   # 👈 APPEND DF

        i += 1


# --- CLEANUP ---
i = 0
while i < len(drivers):

    try:
        if drivers[i]:
            drivers[i].quit()
    except:
        pass

    i += 1


# --- FINAL COMBINE ---
if len(dfs) > 0:
    final_df = pd.concat(dfs, ignore_index=True)
else:
    final_df = pd.DataFrame()



end = time.time()
scrape_time = end-start
print(scrape_time)



#############################################################################################################

164.2292218208313


In [111]:
final_df
final_df.sort_values(by="sent_score", ascending=False).reset_index(drop=True)[:50]

,ticker,watch_count,sent_score,message_volume,participation_ratio
0,PN,1558.0,89.0,99.0,99.0
1,XRX,4106.0,89.0,95.0,95.0
2,ZETA,12679.0,86.0,87.0,87.0
3,CRNT,8066.0,85.0,52.0,52.0
4,RXT,7300.0,84.0,71.0,71.0
5,PHAT,2356.0,84.0,73.0,73.0
6,CCOI,534.0,80.0,85.0,85.0
7,ULCC,2035.0,80.0,71.0,71.0
8,VLN,2093.0,78.0,88.0,88.0
9,CERS,4451.0,78.0,96.0,96.0


In [58]:
#Google News Filter  -- replace with booster not filter
##################################################################################################################
update_loading_page(driver, "Fetching Google News (API Version)")
time.sleep(1)
update_loading_page(driver, f"Fetching Google News (API Version) - {len(filtered_tickers)} stocks ")


#Scrape Google News
google_news_df = run_fetch_google_news(ticker_list = filtered_tickers , period = '12h', workers = 20) #inserts into Google news. 


#Filter Google News
google_news_df = run_sql("""

SELECT 
    ticker,
    COUNT(*) AS article_count,
    MAX(created_at) AS latest_article_time
FROM google_news_links
WHERE created_at >= NOW() - INTERVAL 12 HOUR
GROUP BY ticker
HAVING COUNT(*) > 1
ORDER BY article_count DESC;


""").to_df()

filtered_tickers = google_news_df['ticker'].to_list()
update_loading_page(driver, f"Complete - {len(filtered_tickers)} stocks passed filter ")

##################################################################################################################




🧲 Fetching 1450 Google News RSS...


<string>:2196: DtypeWarning: Columns (34) have mixed types. Specify dtype option on import or set low_memory=False.


🔄 50 / 1450
🔄 100 / 1450
🔄 150 / 1450
🔄 200 / 1450
🔄 250 / 1450
🔄 300 / 1450
🔄 350 / 1450
🔄 400 / 1450
🔄 450 / 1450
🔄 500 / 1450
🔄 550 / 1450
🔄 600 / 1450
🔄 650 / 1450
🔄 700 / 1450
🔄 750 / 1450
🔄 800 / 1450
🔄 850 / 1450
🔄 900 / 1450
🔄 950 / 1450
🔄 1000 / 1450
🔄 1050 / 1450
🔄 1100 / 1450
🔄 1150 / 1450
🔄 1200 / 1450
🔄 1250 / 1450
🔄 1300 / 1450
🔄 1350 / 1450
🔄 1400 / 1450
🔄 1450 / 1450
🧹 Deduped: 4032 → 3865
🧹 Time Filter: 3865 → 1444
🕒 Cutoff: 2026-05-05 09:41:45.760959+00:00
✅ Final rows: 1444
💾 Inserted (or skipped duplicates): 1444 rows


,ticker,message_volume,unique_users,total_likes,net_sentiment,avg_sentiment,participation_ratio,likes_per_message,bullish_percent,signal_score
140,PN,30,28,27.0,21.0,0.700000,0.933333,0.900000,85.000000,272.429652
185,XRX,30,26,56.0,17.0,0.566667,0.866667,1.866667,78.333333,233.129576
15,AVNS,30,27,34.0,15.0,0.500000,0.900000,1.133333,75.000000,231.794136
34,CIFR,30,28,80.0,12.0,0.400000,0.933333,2.666667,70.000000,224.353831
81,GRRR,30,22,176.0,23.0,0.766667,0.733333,5.866667,88.333333,222.446060
...,...,...,...,...,...,...,...,...,...,...
44,CRNT,30,8,29.0,15.0,0.500000,0.266667,0.966667,75.000000,68.679744
61,EVGO,30,11,5.0,2.0,0.066667,0.366667,0.166667,53.333333,67.153528
186,XXI,30,7,48.0,5.0,0.166667,0.233333,1.600000,58.333333,46.740381
171,TONX,30,7,28.0,-10.0,-0.333333,0.233333,0.933333,33.333333,26.708789
